# Iran war / Middle East escalation — Polymarket implied probabilities

Tracks the deepest active Polymarket markets on Iran-related conflict milestones (strike, ceasefire, regime, nuclear, oil). Near-duplicates are collapsed to the single largest-liquidity market per topic.

In [ ]:
import pandas as pd

from alpha_lab.data.loaders.polymarket import (
    search_markets,
    tidy,
    top_by_liquidity,
)

In [ ]:
# Bucket search terms loosely by theme — one deepest market per theme will be kept.
THEMES = {
    "strike": ["iran strike", "israel iran", "attack iran"],
    "ceasefire": ["iran ceasefire", "iran truce"],
    "nuclear": ["iran nuclear", "iran enrichment", "iran bomb"],
    "regime": ["khamenei", "iran regime", "iranian leader"],
    "oil": ["hormuz", "iran oil", "strait of hormuz"],
    "us_involvement": ["us iran war", "trump iran", "us strike iran"],
}

def fetch_theme(queries: list[str]) -> pd.DataFrame:
    frames = [search_markets(q, limit=100) for q in queries]
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["id"])
    return df

theme_markets = {name: fetch_theme(qs) for name, qs in THEMES.items()}
{k: len(v) for k, v in theme_markets.items()}

In [ ]:
# Pick the single largest-liquidity market per theme.
picks = []
for theme, df in theme_markets.items():
    if df.empty:
        continue
    pick = top_by_liquidity(df, n=1).assign(theme=theme)
    picks.append(pick)

dashboard = pd.concat(picks, ignore_index=True) if picks else pd.DataFrame()
dashboard

In [ ]:
# Compact implied-probability view, one row per theme.
view = tidy(dashboard)
view.insert(0, "theme", dashboard["theme"].values)
view